# 06 - Ablation Evaluation for ClawTeam v3.1

This notebook evaluates the central claim of ClawTeam: **harness orchestration adds value, and role LoRA adapters add further value on top of the harness**.

Main groups:
- E1: single-agent baseline
- E2: four specialists, Round 1 only, no LoRA
- E3: four specialists, Round 1 + Round 2 critique, no LoRA
- E4: E3 + Qwen3 LoRA for surgeon and medical oncologist

Important fix: each group explicitly resets LoRA environment flags and clears the in-process LoRA cache, so E2/E3 cannot be polluted by cloud `.env` settings.


## Step 0: 环境准备

In [ ]:
import os, sys, asyncio, json, time
from pathlib import Path
from collections import defaultdict


def find_backend_dir(start: Path) -> Path:
    candidates = [start, *start.parents]
    for base in candidates:
        for candidate in (base, base / 'backend'):
            if (candidate / 'graph').is_dir() and (candidate / 'config').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Cannot locate backend directory from notebook cwd')


NOTEBOOK_DIR = Path.cwd().resolve()
BACKEND_DIR = find_backend_dir(NOTEBOOK_DIR)
EVAL_DIR = BACKEND_DIR / 'eval'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

DATA_DIR = EVAL_DIR / 'datasets' / 'data'
RESULTS_DIR = EVAL_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Backend dir: {BACKEND_DIR}')
print(f'Data dir: {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')


## Step 1: 加载评测集（限制规模 100-200 题）

In [ ]:
# Load oncology evaluation cases.
# TEST_LIMIT controls reading; EVAL_LIMIT controls actual cost.
TEST_LIMIT = int(os.getenv('ABLATION_TEST_LIMIT', '100'))
EVAL_LIMIT = int(os.getenv('ABLATION_EVAL_LIMIT', '30'))


def load_eval_set():
    test_cases = []

    medqa_test = DATA_DIR / 'oncology' / 'medqa_test_oncology.jsonl'
    if medqa_test.exists():
        with open(medqa_test, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                options = item.get('options', [])
                if isinstance(options, list) and options:
                    opt_text = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(options))
                    question_text = f"{item['question']}\n\n{opt_text}"
                else:
                    question_text = item['question']
                test_cases.append({
                    'id': f"MedQA-{len(test_cases)+1}",
                    'source': 'MedQA',
                    'question': question_text,
                    'answer': item.get('answer_text', ''),
                    'answer_idx': item.get('answer_idx'),
                    'options': options,
                })
                if len(test_cases) >= TEST_LIMIT:
                    break

    tumor_board = DATA_DIR / 'tumor_board_cases.jsonl'
    if tumor_board.exists():
        with open(tumor_board, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                test_cases.append({
                    'id': f"TumorBoard-{len(test_cases)+1}",
                    'source': 'TumorBoard',
                    'question': item.get('case', ''),
                    'answer': item.get('expected_answer', ''),
                    'options': [],
                })

    return test_cases[:EVAL_LIMIT]


test_cases = load_eval_set()
mcq_cases = [c for c in test_cases if c.get('options')]
print(f'eval cases: {len(test_cases)}, scored MCQ cases: {len(mcq_cases)}')
print('Tip: set ABLATION_EVAL_LIMIT=100 before launching Jupyter to run a larger eval.')
if test_cases:
    print(f'\nFirst sample:\n{json.dumps(test_cases[0], ensure_ascii=False, indent=2)[:500]}')


## Step 2: 准备评估器 — 4 种实验配置

In [ ]:
import re
import gc
from graph.coordinator import Coordinator
from graph.complexity_assessor import CaseComplexity
from config import get_settings
from graph.llm import build_llm_config_from_settings, get_llm

coordinator = None

LORA_BASE = '/root/autodl-tmp/Qwen3-4B-Instruct-2507'
LORA_SURGEON_PATH = 'eval/models/surgeon_qwen3_lora'
LORA_ONCOLOGIST_PATH = 'eval/models/oncologist_qwen3_lora'


def clear_lora_cache():
    """Clear the LoRA loader cache so experiment groups are isolated."""
    try:
        import eval.inference.load_lora_role as loader
        cache = getattr(loader, '_lora_cache', None)
        if isinstance(cache, dict):
            cache.clear()
    except Exception as exc:
        print(f'[warn] clear_lora_cache failed: {exc}')
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def configure_lora(enabled: bool):
    """Explicitly configure LoRA before each experiment group."""
    os.environ['USE_LORA_SURGEON'] = 'true' if enabled else 'false'
    os.environ['USE_LORA_MEDICAL_ONCOLOGIST'] = 'true' if enabled else 'false'
    os.environ['LORA_SURGEON_BASE'] = LORA_BASE
    os.environ['LORA_SURGEON_PATH'] = LORA_SURGEON_PATH
    os.environ['LORA_MEDICAL_ONCOLOGIST_BASE'] = LORA_BASE
    os.environ['LORA_MEDICAL_ONCOLOGIST_PATH'] = LORA_ONCOLOGIST_PATH
    clear_lora_cache()
    print(f'LoRA enabled: {enabled}')


def reset_coordinator(use_lora: bool):
    global coordinator
    configure_lora(use_lora)
    coordinator = Coordinator(BACKEND_DIR)
    return coordinator


def build_eval_prompt(case: dict) -> str:
    """Normalize E1-E4 input and ask for a parseable final answer."""
    q = case['question'].strip()
    if case.get('options'):
        return (
            f"{q}\n\n"
            "Reason briefly, then end with exactly one final answer line:\n"
            "Final Answer: <A/B/C/D>\n"
            "The final line must contain one option letter only."
        )
    return q


def expected_letter(case: dict) -> str | None:
    options = case.get('options') or []
    idx = case.get('answer_idx')
    if isinstance(idx, int):
        return chr(65 + idx) if 0 <= idx < 26 else None
    if isinstance(idx, str) and idx.strip():
        s = idx.strip().upper()
        if len(s) == 1 and 'A' <= s <= 'Z':
            return s
        if s.isdigit():
            n = int(s)
            if 0 <= n < 26:
                return chr(65 + n)
    expected = str(case.get('answer', '')).strip().lower()
    for i, option in enumerate(options):
        if str(option).strip().lower() == expected:
            return chr(65 + i)
    return None


def extract_final_letter(predicted: str, options: list | None = None) -> str | None:
    """Parse only explicit final answers; avoid broad substring matches."""
    if not predicted:
        return None
    text = predicted.strip()
    patterns = [
        r'final\s*answer\s*[:\uff1a]\s*([A-H])\b',
        r'\u6700\u7ec8\u7b54\u6848\s*[:\uff1a]?\s*([A-H])\b',
        r'\u7b54\u6848\s*[:\uff1a]?\s*([A-H])\b',
        r'\u9009\u62e9\s*([A-H])\b',
        r'\u9009\u9879\s*([A-H])\b',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            return matches[-1].upper()

    tail = text[-500:]
    matches = re.findall(r'(?<![A-Za-z])([A-H])\s*[\.\u3001\)]', tail)
    if matches:
        return matches[-1].upper()
    return None


async def run_e1_single_agent(case):
    settings = get_settings()
    llm = get_llm(build_llm_config_from_settings(settings, temperature=0.3, streaming=False))
    response = await llm.ainvoke([
        {'role': 'system', 'content': 'You are a medical exam assistant. Answer oncology MCQs and end with Final Answer: <letter>.'},
        {'role': 'user', 'content': build_eval_prompt(case)},
    ])
    return getattr(response, 'content', '').strip(), None


async def run_e2_no_debate(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.MODERATE,
        skip_round2=True,
    )
    return session.final_decision, session


async def run_e3_with_debate(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.COMPLEX,
        skip_round2=False,
    )
    return session.final_decision, session


async def run_e4_full(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.COMPLEX,
        skip_round2=False,
    )
    return session.final_decision, session


EXPERIMENTS = {
    'E1': {'name': 'E1_single', 'config': 'single-agent baseline', 'use_lora': False, 'fn': run_e1_single_agent},
    'E2': {'name': 'E2_no_debate', 'config': '4 specialists, Round 1 only, no LoRA', 'use_lora': False, 'fn': run_e2_no_debate},
    'E3': {'name': 'E3_debate', 'config': '4 specialists + Round 2 critique, no LoRA', 'use_lora': False, 'fn': run_e3_with_debate},
    'E4': {'name': 'E4_lora', 'config': 'E3 + surgeon/oncologist LoRA', 'use_lora': True, 'fn': run_e4_full},
}

print('Experiment setup loaded: isolated LoRA flags, strict answer parsing, detailed logs.')


## Step 3: Evaluation helpers - strict answer parsing and detailed logs


In [ ]:
def score_prediction(predicted: str, case: dict) -> dict:
    exp = expected_letter(case)
    pred = extract_final_letter(predicted, case.get('options'))
    scored = exp is not None
    return {
        'expected_letter': exp,
        'predicted_letter': pred,
        'scored': scored,
        'correct': bool(scored and pred == exp),
    }


async def evaluate_method(exp_key: str, cases: list[dict]):
    """Run one experiment group and keep per-case details."""
    spec = EXPERIMENTS[exp_key]
    reset_coordinator(use_lora=spec['use_lora'])

    correct = 0
    scored_total = 0
    latencies = []
    sessions = []
    details = []

    for i, case in enumerate(cases):
        start = time.monotonic()
        result = ''
        session = None
        error = ''
        try:
            result, session = await spec['fn'](case)
            if session is not None:
                sessions.append(session)
        except Exception as exc:
            error = repr(exc)
            print(f'  [{i+1}/{len(cases)}] {spec["name"]} failed: {exc}')
        elapsed = time.monotonic() - start
        latencies.append(elapsed)

        score = score_prediction(result, case)
        if score['scored']:
            scored_total += 1
            correct += int(score['correct'])

        details.append({
            'method': exp_key,
            'method_name': spec['name'],
            'case_id': case.get('id', i + 1),
            'source': case.get('source', ''),
            'expected_letter': score['expected_letter'],
            'predicted_letter': score['predicted_letter'],
            'correct': score['correct'],
            'scored': score['scored'],
            'latency_s': elapsed,
            'error': error,
            'question': case.get('question', '')[:500],
            'answer': case.get('answer', ''),
            'prediction': result,
        })

        if (i + 1) % 5 == 0 or i + 1 == len(cases):
            acc = correct / scored_total if scored_total else 0
            print(f'  [{i+1}/{len(cases)}] {spec["name"]}: scored={scored_total}, acc={acc:.3f}')

    revision_rates = [s.revision_rate for s in sessions if getattr(s, 'round2_opinions', None)]
    disagreement_counts = [s.disagreement_count for s in sessions if getattr(s, 'round2_opinions', None)]

    return {
        'method': spec['name'],
        'config': spec['config'],
        'total': len(cases),
        'scored_total': scored_total,
        'correct': correct,
        'accuracy': correct / scored_total if scored_total else 0,
        'avg_latency_s': sum(latencies) / len(latencies) if latencies else 0,
        'sessions': sessions,
        'details': details,
        'avg_revision_rate': sum(revision_rates) / len(revision_rates) if revision_rates else 0,
        'avg_disagreement_count': sum(disagreement_counts) / len(disagreement_counts) if disagreement_counts else 0,
    }


## Step 4: 跑主实验（4 组）

In [ ]:
# Main 4-group experiment. Each group resets LoRA flags and cache.
results = {}
all_details = []

for exp_key in ['E1', 'E2', 'E3', 'E4']:
    print(f'=== {exp_key}: {EXPERIMENTS[exp_key]["config"]} ===')
    results[exp_key] = await evaluate_method(exp_key, test_cases)
    all_details.extend(results[exp_key]['details'])
    print(
        f'{exp_key} accuracy: {results[exp_key]["accuracy"]:.3f} '
        f'({results[exp_key]["correct"]}/{results[exp_key]["scored_total"]}), '
        f'latency: {results[exp_key]["avg_latency_s"]:.2f}s\n'
    )


<!-- merged into Step 4 runner above -->


<!-- merged into Step 4 runner above -->


<!-- merged into Step 4 runner above -->


## Step 5: 主实验汇总表（论文 Table 1）

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'experiment': k,
        'config': v['config'],
        'scored_cases': v['scored_total'],
        'correct': v['correct'],
        'accuracy': f"{v['accuracy']:.3f}",
        'avg_latency_s': f"{v['avg_latency_s']:.2f}",
        'avg_revision_rate': f"{v['avg_revision_rate']:.3f}",
        'avg_disagreement_count': f"{v['avg_disagreement_count']:.2f}",
        'gain_vs_E1': f"{(v['accuracy'] - results['E1']['accuracy']) * 100:+.1f}%",
    }
    for k, v in results.items()
])
print('\n=== Main ablation results ===')
print(summary.to_string(index=False))

summary.to_csv(RESULTS_DIR / 'main_ablation.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(all_details).to_csv(RESULTS_DIR / 'main_ablation_details.csv', index=False, encoding='utf-8-sig')
print(f'\nSaved: {RESULTS_DIR / "main_ablation.csv"}')
print(f'Details: {RESULTS_DIR / "main_ablation_details.csv"}')


## Step 6: 子消融 1 — 训练 vs 不训练（外科 / 内科）

In [ ]:
print('=== Sub-ablation 1: training vs no training ===')
print('Use E4 - E3 as the incremental value of surgeon/oncologist LoRA within the same harness.')
training_gain = results['E4']['accuracy'] - results['E3']['accuracy']
print(f'Training gain (E4 - E3): {training_gain * 100:+.1f}%')
print('Note: LoRA is used only by surgeon and medical_oncologist. Pathologist and radiation_oncologist stay API + prompt.')


## Step 7: 子消融 2 — 复杂度评估方法对比

In [ ]:
from graph.complexity_assessor import assess_complexity

print('=== Complexity assessor comparison: distribution only ===')
routing_rows = []
for method in ['llm', 'bert', 'keyword']:
    counts = defaultdict(int)
    ok = 0
    for case in test_cases[:30]:
        try:
            decision = await assess_complexity(case['question'], method=method)
            counts[decision.level.value] += 1
            ok += 1
        except Exception as exc:
            counts['error'] += 1
            print(f'  {method} failed: {exc}')
    routing_rows.append({'method': method, 'ok': ok, **dict(counts)})

routing_df = pd.DataFrame(routing_rows).fillna(0)
print(routing_df.to_string(index=False))
routing_df.to_csv(RESULTS_DIR / 'complexity_methods.csv', index=False, encoding='utf-8-sig')
print('No gold complexity labels are available, so this is not reported as routing accuracy.')


## Step 8: 子消融 3 — Round 数 acc vs cost

In [ ]:
round_comp = pd.DataFrame([
    {
        'rounds': '1 (independent)',
        'experiment': 'E2',
        'accuracy': f"{results['E2']['accuracy']:.3f}",
        'avg_latency_s': f"{results['E2']['avg_latency_s']:.2f}",
        'avg_revision_rate': '-',
        'relative_cost': '1x',
    },
    {
        'rounds': '2 (independent + critique)',
        'experiment': 'E3',
        'accuracy': f"{results['E3']['accuracy']:.3f}",
        'avg_latency_s': f"{results['E3']['avg_latency_s']:.2f}",
        'avg_revision_rate': f"{results['E3']['avg_revision_rate']:.3f}",
        'relative_cost': f'{results["E3"]["avg_latency_s"] / max(results["E2"]["avg_latency_s"], 1e-6):.1f}x',
    },
])
print('\n=== Round count comparison ===')
print(round_comp.to_string(index=False))
round_comp.to_csv(RESULTS_DIR / 'round_comparison.csv', index=False, encoding='utf-8-sig')


## Step 9: 论文用图表（matplotlib）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(10, 6))
names = ['E1\nSingle', 'E2\nRoles', 'E3\nDebate', 'E4\nLoRA']
accs = [results[k]['accuracy'] for k in ['E1', 'E2', 'E3', 'E4']]
colors = ['#8A8F98', '#3B82F6', '#F59E0B', '#10B981']
bars = ax.bar(names, accs, color=colors)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f'{acc:.3f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy on scored MCQ cases', fontsize=12)
ax.set_title('ClawTeam Main Ablation: Harness and LoRA Contributions', fontsize=14)
ax.set_ylim(0, max(max(accs) * 1.2, 0.1))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'main_ablation.png', dpi=150)
plt.show()
print(f'\nFigure saved: {RESULTS_DIR / "main_ablation.png"}')


## Step 10: 总结报告（论文用）

In [ ]:
report = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'eval_set_size': len(test_cases),
    'scored_mcq_size': len([c for c in test_cases if expected_letter(c)]),
    'main_ablation': {
        k: {
            'config': v['config'],
            'accuracy': v['accuracy'],
            'correct': v['correct'],
            'scored_total': v['scored_total'],
            'avg_latency_s': v['avg_latency_s'],
            'avg_revision_rate': v['avg_revision_rate'],
            'avg_disagreement_count': v['avg_disagreement_count'],
        }
        for k, v in results.items()
    },
    'key_findings': {
        'role_decomposition_gain_E2_vs_E1': results['E2']['accuracy'] - results['E1']['accuracy'],
        'debate_gain_E3_vs_E2': results['E3']['accuracy'] - results['E2']['accuracy'],
        'lora_gain_E4_vs_E3': results['E4']['accuracy'] - results['E3']['accuracy'],
        'total_gain_E4_vs_E1': results['E4']['accuracy'] - results['E1']['accuracy'],
    },
    'caveats': [
        'Accuracy is computed only on MCQ cases with an extractable final answer letter.',
        'TumorBoard free-form cases should be evaluated by rubric or LLM judge separately.',
        'Complexity assessor comparison reports distribution only because no gold complexity labels are available.',
    ],
}

with open(RESULTS_DIR / 'final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
